# Fine-tune Qwen3-8B on SDET Playwright Test Generation with Unsloth

Train a small 8B model to generate Playwright tests using the compiled SDET workflow.

## Setup
1. Upload `training_unsloth.jsonl` to your Google Drive
2. Run all cells below
3. The fine-tuned model will be saved to your Drive

In [ ]:
# Install Unsloth (takes ~2 min on Colab)
# Use --no-deps to avoid backtracking hell with Colab's pre-installed torch 2.11
import torch
major_version, minor_version = torch.__version__.split(".")[:2]
major_version, minor_version = int(major_version), int(minor_version)

unsloth_pkg = "unsloth[cu124-torch250]" if (major_version >= 2 and minor_version >= 5) else "unsloth[cu121-torch240]"
!pip install "{unsloth_pkg} @ git+https://github.com/unslothai/unsloth.git" --no-deps -q
!pip install unsloth_zoo bitsandbytes xformers trl peft accelerate wandb datasets -q

In [ ]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
from datasets import Dataset, DatasetDict
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, standardize_sharegpt, train_on_responses_only
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

DATA_PATH = "/content/drive/MyDrive/training_unsloth.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/qwen3-8b-sdet"
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_SEQ_LENGTH = 8192
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR = 2e-5
NUM_EPOCHS = 3

In [ ]:
# Load ShareGPT dataset
raw = []
with open(DATA_PATH) as f:
    for line in f:
        raw.append(json.loads(line))

# Strip metadata — Unsloth only needs 'conversations'
data = [{"conversations": item["conversations"]} for item in raw]
dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.01, seed=42)
print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

In [ ]:
# Load Qwen3-8B with Unsloth's 4-bit QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
# Apply Qwen's chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-3",  # or "chatml" for Qwen3
)

# Standardize ShareGPT fields → 'from'/'value' format
dataset = standardize_sharegpt(dataset)

# Format conversations into single text with chat template
def format_fn(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": texts}

dataset = dataset.map(format_fn, batched=True)
print(dataset["train"]["text"][0][:300])

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# Training arguments
args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=200,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
    output_dir=OUTPUT_DIR,
    report_to="none",
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    seed=42,
)

In [ ]:
# Create SFT trainer with response-only masking
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)

# Mask user inputs so loss is only computed on assistant responses
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Train! (approx 2-3 hours on a T4, ~45 min on A100)
train_stats = trainer.train()
print(f"Training complete. Loss: {train_stats.training_loss:.4f}")

In [ ]:
# Save LoRA adapters to Drive
model.save_pretrained_merged(OUTPUT_DIR, tokenizer, save_method="lora")
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "Write a Playwright test for the login page at https://app.example.com/login"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt", padding=True).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(response)